# Classes Are Callable — Advanced Tutorial Problems with Solutions

This notebook develops the topic **Classes Are Callable** in a tutorial style.

Rather than jumping directly to large exercises, each section follows a teaching rhythm:

1. introduce one idea;
2. run a small experiment;
3. inspect what Python did;
4. make a prediction;
5. solve a focused problem;
6. explain the result;
7. extend the idea.

The notebook starts from ordinary class calls and gradually reaches `__new__`, callable instances, alternate constructors, dynamic classes, metaclasses, caching, registries, and callable pipelines.

## Starting mental model

When we write:

```python
obj = MyClass(...)
```

we are **calling the class object**.

For an ordinary user-defined class, that call usually creates and initializes a new instance.

In [1]:
class Example:
    pass

print("callable(Example):", callable(Example))

obj = Example()

print("type(obj):", type(obj))
print("isinstance(obj, Example):", isinstance(obj, Example))

callable(Example): True
type(obj): <class '__main__.Example'>
isinstance(obj, Example): True


The class object `Example` is callable.

The result of calling it is an instance whose type is `Example`.

We will now build on this one idea step by step.

# Tutorial Problem 1 — Class namespace vs instance namespace

A class can contain data shared by every instance, while each instance can also contain its own data.

We will inspect both namespaces directly.

In [2]:
class Language:
    category = "programming language"

    def __init__(self, name, year):
        self.name = name
        self.year = year

At this point the class already exists, even though no instance has been created.

Let us inspect two entries from the class namespace.

In [3]:
print("category:", Language.__dict__["category"])
print("__init__:", Language.__dict__["__init__"])

category: programming language
__init__: <function Language.__init__ at 0x00000167792FD260>


Now create an instance.

Before running the next cell, predict which attributes will appear in `python.__dict__`.

In [4]:
python = Language("Python", 1991)

print(python.__dict__)

{'name': 'Python', 'year': 1991}


The instance contains only the values assigned to `self`.

The class attribute `category` is not inside the instance dictionary, but normal attribute lookup can still find it.

In [5]:
print("python.category:", python.category)
print("'category' in python.__dict__:", "category" in python.__dict__)

python.category: programming language
'category' in python.__dict__: False


### Problem

Create a second `Language` instance for JavaScript and prove that:

- the instances have separate `name` and `year` values;
- both obtain `category` from the class;
- neither stores `category` in its own `__dict__`.

In [6]:
# Solution

javascript = Language("JavaScript", 1995)

print("python:", python.__dict__)
print("javascript:", javascript.__dict__)

assert python.name == "Python"
assert javascript.name == "JavaScript"
assert python.year == 1991
assert javascript.year == 1995

assert python.category == "programming language"
assert javascript.category == "programming language"

assert "category" not in python.__dict__
assert "category" not in javascript.__dict__

python: {'name': 'Python', 'year': 1991}
javascript: {'name': 'JavaScript', 'year': 1995}


The important distinction is:

```text
class namespace    -> shared attributes and behavior
instance namespace -> state belonging to one object
```

# Tutorial Problem 2 — Attribute shadowing

Suppose a class provides a default value.

An instance can define an attribute with the same name. That instance-level value then shadows the class attribute for that object.

In [7]:
class Settings:
    mode = "production"

a = Settings()
b = Settings()

print(a.mode)
print(b.mode)

production
production


Both values currently come from the class.

Now assign a value only to `a`.

In [8]:
a.mode = "development"

print("a.__dict__:", a.__dict__)
print("b.__dict__:", b.__dict__)

a.__dict__: {'mode': 'development'}
b.__dict__: {}


Predict the output before running:

In [9]:
print("a.mode:", a.mode)
print("b.mode:", b.mode)

a.mode: development
b.mode: production


Now change the class default itself.

In [10]:
Settings.mode = "testing"

print("a.mode:", a.mode)
print("b.mode:", b.mode)

a.mode: development
b.mode: testing


### Problem

Use namespace inspection to explain why `a` and `b` now report different values.

In [11]:
# Solution

print("'mode' in a.__dict__:", "mode" in a.__dict__)
print("'mode' in b.__dict__:", "mode" in b.__dict__)
print("'mode' in Settings.__dict__:", "mode" in Settings.__dict__)

assert a.mode == "development"
assert b.mode == "testing"

'mode' in a.__dict__: True
'mode' in b.__dict__: False
'mode' in Settings.__dict__: True


`a` has its own `mode`, so lookup stops there.

`b` has no instance-level `mode`, so Python reaches `Settings.mode`.

# Tutorial Problem 3 — Removing a shadow

If an instance attribute is deleted, the class-level value becomes visible again.

In [12]:
print("before deletion:", a.mode)

del a.mode

print("after deletion:", a.mode)
print("a.__dict__:", a.__dict__)

before deletion: development
after deletion: testing
a.__dict__: {}


### Problem

Reset the class default to `"production"`.

Then:

1. create a fresh instance;
2. shadow the value;
3. delete the shadow;
4. assert the value at every stage.

In [13]:
# Solution

Settings.mode = "production"

c = Settings()
assert c.mode == "production"

c.mode = "local"
assert c.mode == "local"
assert c.__dict__["mode"] == "local"

del c.mode

assert c.mode == "production"
assert "mode" not in c.__dict__

print("final value:", c.mode)

final value: production


This is a useful way to reason about many attribute-lookup puzzles: ask **where the attribute currently lives**.

# Tutorial Problem 4 — `type()` and `isinstance()`

For ordinary objects, `type(obj)` gives the exact runtime type.

`isinstance(obj, SomeClass)` answers a slightly different question: whether the object belongs to that class or one of its subclasses.

In [14]:
class Worker:
    pass

class Manager(Worker):
    pass

w = Worker()
m = Manager()

print("type(w):", type(w))
print("type(m):", type(m))

type(w): <class '__main__.Worker'>
type(m): <class '__main__.Manager'>


Now compare exact type checks with inheritance-aware checks.

In [15]:
print("type(m) is Worker:", type(m) is Worker)
print("type(m) is Manager:", type(m) is Manager)
print("isinstance(m, Worker):", isinstance(m, Worker))
print("isinstance(m, Manager):", isinstance(m, Manager))

type(m) is Worker: False
type(m) is Manager: True
isinstance(m, Worker): True
isinstance(m, Manager): True


### Problem

Write two functions:

- one that accepts the whole `Worker` family;
- one that accepts only exact `Worker` instances.

In [16]:
# Solution

def accepts_worker_family(obj):
    return isinstance(obj, Worker)

def accepts_exact_worker(obj):
    return type(obj) is Worker

assert accepts_worker_family(w)
assert accepts_worker_family(m)

assert accepts_exact_worker(w)
assert not accepts_exact_worker(m)

print("All checks passed.")

All checks passed.


Best practice:

- use `isinstance()` for most polymorphic code;
- use exact `type(...) is ...` checks only when subclasses must be intentionally excluded.

# Tutorial Problem 5 — `type(obj)` vs `obj.__class__`

Usually `obj.__class__` appears to agree with `type(obj)`.

In [17]:
class Normal:
    pass

n = Normal()

print(type(n))
print(n.__class__)
print(type(n) is n.__class__)

<class '__main__.Normal'>
<class '__main__.Normal'>
True


But `__class__` can participate in normal attribute lookup.

That means unusual class definitions can make direct access misleading.

In [18]:
class Strange:
    __class__ = str

s = Strange()

print("type(s):", type(s))
print("s.__class__:", s.__class__)

type(s): <class '__main__.Strange'>
s.__class__: <class 'str'>


### Problem

Prove that the actual runtime type remains `Strange`.

In [19]:
# Solution

assert type(s) is Strange
assert s.__class__ is str

print("type() reports the actual runtime type.")

type() reports the actual runtime type.


This example is intentionally unusual.

For exact runtime type inspection, prefer `type(obj)` rather than relying on manually accessed `obj.__class__`.

# Tutorial Problem 6 — Classes are only one kind of callable

Python's `callable()` asks a broad question:

> Can this object be invoked with parentheses?

Functions are callable. Classes are callable. Some instances are callable too.

In [20]:
def greet(name):
    return f"Hello, {name}"

class User:
    pass

print("function:", callable(greet))
print("class:", callable(User))
print("integer:", callable(42))

function: True
class: True
integer: False


A plain instance is usually not callable.

In [21]:
u = User()
print("User instance:", callable(u))

User instance: False


To make an instance callable, define `__call__`.

In [22]:
class Greeter:
    def __init__(self, prefix):
        self.prefix = prefix

    def __call__(self, name):
        return f"{self.prefix}, {name}!"

friendly = Greeter("Hi")

print(callable(friendly))
print(friendly("Ada"))

True
Hi, Ada!


### Problem

Create a callable class `Power(exponent)`.

The resulting instance should behave like a configured function.

In [23]:
# Solution

class Power:
    def __init__(self, exponent):
        self.exponent = exponent

    def __call__(self, value):
        return value ** self.exponent

square = Power(2)
cube = Power(3)

print(square(5))
print(cube(5))

assert square(5) == 25
assert cube(5) == 125
assert callable(square)
assert callable(cube)

25
125


Callable objects are useful when function-like behavior needs persistent state or configuration.

# Tutorial Problem 7 — Stateful callable objects

A callable object can remember information between calls.

This makes it different from a simple stateless function.

In [24]:
class CallCounter:
    def __init__(self):
        self.count = 0

    def __call__(self, value):
        self.count += 1
        return f"call {self.count}: {value}"

counter = CallCounter()

print(counter("alpha"))
print(counter("beta"))
print(counter("gamma"))

call 1: alpha
call 2: beta
call 3: gamma


The object behaves like a function, but `count` remains attached to the same instance.

### Problem

Create a callable `Clamp(min_value, max_value)`.

It should force values into the configured inclusive range.

In [25]:
# Solution

class Clamp:
    def __init__(self, min_value, max_value):
        if min_value > max_value:
            raise ValueError("min_value cannot exceed max_value")

        self.min_value = min_value
        self.max_value = max_value

    def __call__(self, value):
        if value < self.min_value:
            return self.min_value

        if value > self.max_value:
            return self.max_value

        return value

percentage = Clamp(0, 100)

for value in (-20, 0, 25, 100, 150):
    print(value, "->", percentage(value))

assert percentage(-20) == 0
assert percentage(25) == 25
assert percentage(150) == 100

-20 -> 0
0 -> 0
25 -> 25
100 -> 100
150 -> 100


This is a good example of an object acting as a **configured function**.

# Tutorial Problem 8 — Follow `__new__` and `__init__`

A normal class call involves two different responsibilities:

```text
__new__  -> produce the object
__init__ -> initialize the object
```

Let us observe the order directly.

In [26]:
class Traced:
    def __new__(cls, value):
        print("__new__ received:", cls.__name__, value)

        obj = super().__new__(cls)

        print("__new__ created id:", id(obj))

        return obj

    def __init__(self, value):
        print("__init__ received id:", id(self), value)

        self.value = value

t = Traced(10)

__new__ received: Traced 10
__new__ created id: 1543925856448
__init__ received id: 1543925856448 10


The same object created by `__new__` is passed into `__init__`.

In [27]:
print(t.__dict__)
print(t.value)

{'value': 10}
10


### Problem

Create `TrackedPoint(x, y)`.

Store the object ID during `__new__`, then verify inside `__init__` that it is the same object.

In [28]:
# Solution

class TrackedPoint:
    def __new__(cls, x, y):
        obj = super().__new__(cls)
        obj.created_id = id(obj)

        print("__new__ id:", obj.created_id)

        return obj

    def __init__(self, x, y):
        print("__init__ id:", id(self))

        assert self.created_id == id(self)

        self.x = x
        self.y = y

point = TrackedPoint(3, 4)

assert point.x == 3
assert point.y == 4

__new__ id: 1543925857120
__init__ id: 1543925857120


In normal classes, `__new__` creates the object and `__init__` configures its state.

# Tutorial Problem 9 — `__init__` must return `None`

`__init__` initializes an existing instance.

It is not allowed to return a replacement result.

In [29]:
class BadInitializer:
    def __init__(self):
        return "not allowed"

try:
    BadInitializer()
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)

TypeError: __init__() should return None, not 'str'


### Problem

Write a correct `Account` class that validates a non-empty username and stores it.

In [30]:
# Solution

class Account:
    def __init__(self, username):
        if not username:
            raise ValueError("username cannot be empty")

        self.username = username

account = Account("alice")

print(account.__dict__)

assert account.username == "alice"

{'username': 'alice'}


For most ordinary classes, `__init__` is the right place for validation and instance setup.

# Tutorial Problem 10 — What if `__new__` returns another type?

`__new__` can return something other than a new instance of the requested class.

This changes the normal construction flow.

In [31]:
class ReturnsList:
    def __new__(cls):
        print("__new__ is returning a list")
        return ["surprise"]

    def __init__(self):
        print("__init__ ran")

value = ReturnsList()

print(value)
print(type(value))

__new__ is returning a list
['surprise']
<class 'list'>


`__init__` did not run because the object returned by `__new__` was not an instance of `ReturnsList`.

### Problem

Create `MaybeInt`.

- if the input is a genuine integer, return it directly;
- otherwise create a normal `MaybeInt` instance and store the value.

In [32]:
# Solution

class MaybeInt:
    def __new__(cls, value):
        if isinstance(value, int) and not isinstance(value, bool):
            return value

        return super().__new__(cls)

    def __init__(self, value):
        self.value = value

a = MaybeInt(10)
b = MaybeInt("10")

print(a, type(a))
print(b, type(b), b.__dict__)

assert a == 10
assert type(a) is int

assert isinstance(b, MaybeInt)
assert b.value == "10"

10 <class 'int'>
<__main__.MaybeInt object at 0x000001677926E900> <class '__main__.MaybeInt'> {'value': '10'}


This works, but it is surprising.

A named factory function is often clearer when construction may return unrelated types.

# Tutorial Problem 11 — Immutable built-ins and `__new__`

Immutable objects such as `tuple` and `str` cannot be filled in by mutating their contents after creation.

That is why subclasses of immutable built-ins often customize `__new__`.

In [33]:
class Pair(tuple):
    def __new__(cls, left, right):
        return super().__new__(cls, (left, right))

p = Pair("A", "B")

print(p)
print(type(p))

('A', 'B')
<class '__main__.Pair'>


The tuple contents already exist when the object comes back from `__new__`.

### Problem

Create a subclass of `str` called `Slug`.

Construction should:

1. strip surrounding whitespace;
2. convert to lowercase;
3. replace spaces with hyphens.

In [34]:
# Solution

class Slug(str):
    def __new__(cls, value):
        if not isinstance(value, str):
            raise TypeError("value must be a string")

        normalized = value.strip().lower().replace(" ", "-")

        return super().__new__(cls, normalized)

slug = Slug("  Advanced Python Classes  ")

print(slug)
print(type(slug))

assert slug == "advanced-python-classes"
assert isinstance(slug, Slug)
assert isinstance(slug, str)

advanced-python-classes
<class '__main__.Slug'>


The normalization has to happen before the immutable string object is finalized.

# Tutorial Problem 12 — Alternate constructors with `@classmethod`

Sometimes a class has more than one natural construction format.

A class method can provide a named alternative.

In [35]:
class Date:
    def __init__(self, year, month, day):
        self.year = year
        self.month = month
        self.day = day

    @classmethod
    def from_iso(cls, text):
        year, month, day = map(int, text.split("-"))

        return cls(year, month, day)

d = Date.from_iso("2026-09-11")

print(d.__dict__)

{'year': 2026, 'month': 9, 'day': 11}


The key line is:

```python
return cls(...)
```

Using `cls` allows subclasses to inherit the alternate constructor while still constructing themselves.

### Problem

Create `Temperature` with two construction paths:

```python
Temperature(celsius)
Temperature.from_fahrenheit(fahrenheit)
```

Use:

```text
C = (F - 32) * 5 / 9
```

In [36]:
# Solution

class Temperature:
    def __init__(self, celsius):
        self.celsius = float(celsius)

    @classmethod
    def from_fahrenheit(cls, fahrenheit):
        celsius = (fahrenheit - 32) * 5 / 9

        return cls(celsius)

freezing = Temperature.from_fahrenheit(32)
boiling = Temperature.from_fahrenheit(212)

print(freezing.celsius)
print(boiling.celsius)

assert freezing.celsius == 0
assert boiling.celsius == 100

0.0
100.0


A named alternate constructor is often clearer than forcing several unrelated parsing rules into one complicated `__init__`.

# Tutorial Problem 13 — Mutable class attributes can be dangerous

Class attributes are shared.

That is useful when sharing is intentional, but dangerous when the value is mutable and each instance is expected to own its own copy.

In [37]:
class BadCart:
    items = []

    def add(self, item):
        self.items.append(item)

cart1 = BadCart()
cart2 = BadCart()

cart1.add("book")

print("cart1:", cart1.items)
print("cart2:", cart2.items)

cart1: ['book']
cart2: ['book']


Both objects see the same list because the list belongs to the class.

### Problem

Fix the design so each cart has an independent list.

In [38]:
# Solution

class Cart:
    def __init__(self):
        self.items = []

    def add(self, item):
        self.items.append(item)

cart1 = Cart()
cart2 = Cart()

cart1.add("book")
cart2.add("keyboard")

print("cart1:", cart1.items)
print("cart2:", cart2.items)

assert cart1.items == ["book"]
assert cart2.items == ["keyboard"]
assert cart1.items is not cart2.items

cart1: ['book']
cart2: ['keyboard']


Best practice:

Mutable per-instance state usually belongs in `__init__`.

# Tutorial Problem 14 — Not every instance has `__dict__`

Many ordinary Python instances store attributes in an instance dictionary.

But this is not guaranteed.

In [39]:
class CompactPoint:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y

cp = CompactPoint(2, 5)

print(cp.x, cp.y)
print("has __dict__:", hasattr(cp, "__dict__"))

2 5
has __dict__: False


The attributes still exist, but there is no normal instance `__dict__`.

### Problem

Try assigning `cp.z = 10`.

Catch and print the error.

In [40]:
# Solution

try:
    cp.z = 10
except AttributeError as exc:
    print(type(exc).__name__ + ":", exc)

AttributeError: 'CompactPoint' object has no attribute 'z' and no __dict__ for setting new attributes


General-purpose introspection code should not assume that every object stores its state in `obj.__dict__`.

# Tutorial Problem 15 — Methods are callables too

A method defined in a class begins as a function object stored in the class namespace.

When accessed through an instance, it becomes a bound method.

In [41]:
class Calculator:
    def add(self, x, y):
        return x + y

calc = Calculator()

raw = Calculator.__dict__["add"]
through_class = Calculator.add
through_instance = calc.add

print("raw:", raw)
print("through class:", through_class)
print("through instance:", through_instance)

raw: <function Calculator.add at 0x00000167793123E0>
through class: <function Calculator.add at 0x00000167793123E0>
through instance: <bound method Calculator.add of <__main__.Calculator object at 0x0000016779360980>>


Now inspect their types.

In [42]:
print("raw type:", type(raw))
print("class access type:", type(through_class))
print("instance access type:", type(through_instance))

raw type: <class 'function'>
class access type: <class 'function'>
instance access type: <class 'method'>


All of them are callable.

In [43]:
print(callable(raw))
print(callable(through_class))
print(callable(through_instance))

True
True
True


The raw function does not yet have an instance bound to it, so we pass one explicitly.

In [44]:
print(raw(calc, 2, 3))
print(calc.add(2, 3))

5
5


### Problem

Create `Counter.increment()`.

Retrieve the raw function from the class dictionary and invoke it manually with an instance.

In [45]:
# Solution

class Counter:
    def __init__(self):
        self.value = 0

    def increment(self, amount=1):
        self.value += amount
        return self.value

counter = Counter()

raw_increment = Counter.__dict__["increment"]

print(raw_increment(counter, 5))
print(counter.increment(2))

assert counter.value == 7

5
7


This behavior comes from Python's descriptor machinery: function objects stored on a class can produce bound methods when accessed through an instance.

# Tutorial Problem 16 — `type` can create classes dynamically

We have already seen that most ordinary class objects have type `type`.

The built-in `type` can also create a class when called with three arguments:

```python
type(name, bases, namespace)
```

In [46]:
Dynamic = type(
    "Dynamic",
    (),
    {
        "answer": 42,
    },
)

print(Dynamic)
print(type(Dynamic))
print(Dynamic.answer)

<class '__main__.Dynamic'>
<class 'type'>
42


The result is a class object.

Because it is a class, it is itself callable.

In [47]:
dynamic_instance = Dynamic()

print(dynamic_instance)
print(type(dynamic_instance))
print(isinstance(dynamic_instance, Dynamic))

<class '__main__.Dynamic'>
True


### Problem

Dynamically create `DynamicGreeter` with:

- class attribute `prefix = "Hello"`;
- method `greet(self, name)`.

In [48]:
# Solution

def dynamic_greet(self, name):
    return f"{self.prefix}, {name}!"

DynamicGreeter = type(
    "DynamicGreeter",
    (),
    {
        "prefix": "Hello",
        "greet": dynamic_greet,
    },
)

g = DynamicGreeter()

print(g.greet("Ada"))

assert g.greet("Ada") == "Hello, Ada!"
assert type(DynamicGreeter) is type

Hello, Ada!


This is conceptually similar to what a normal `class` statement eventually produces: a class object with a name, bases, and namespace.

# Tutorial Problem 17 — The class behind the class

A metaclass is the type of a class object.

For most user-defined classes, the metaclass is `type`.

In [49]:
class Ordinary:
    pass

print("type(Ordinary):", type(Ordinary))
print("isinstance(Ordinary, type):", isinstance(Ordinary, type))

type(Ordinary): <class 'type'>
isinstance(Ordinary, type): True


We can subclass `type` to create a custom metaclass.

One especially interesting hook is the metaclass's `__call__`, because it runs when the class object itself is called.

In [50]:
class LoggingMeta(type):
    def __call__(cls, *args, **kwargs):
        print("About to call:", cls.__name__)
        print("args:", args)
        print("kwargs:", kwargs)

        result = super().__call__(*args, **kwargs)

        print("Created:", result)

        return result

Now attach the metaclass to an ordinary-looking class.

In [51]:
class Job(metaclass=LoggingMeta):
    def __init__(self, name, priority="normal"):
        self.name = name
        self.priority = priority

    def __repr__(self):
        return f"Job({self.name!r}, priority={self.priority!r})"

job = Job("deploy", priority="high")

About to call: Job
args: ('deploy',)
kwargs: {'priority': 'high'}
Created: Job('deploy', priority='high')


The metaclass intercepted the class call, delegated to normal construction, and then received the created object.

### Problem

Create a metaclass `CountingMeta` that records how many successful constructions have completed.

In [52]:
# Solution

class CountingMeta(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)

        cls.instances_created = 0

        return cls

    def __call__(cls, *args, **kwargs):
        obj = super().__call__(*args, **kwargs)

        cls.instances_created += 1

        return obj

class Task(metaclass=CountingMeta):
    def __init__(self, title):
        if not title:
            raise ValueError("title cannot be empty")

        self.title = title

Task("one")
Task("two")

print(Task.instances_created)

assert Task.instances_created == 2

2


Now try a failed construction.

Because the increment happens after `super().__call__()` returns successfully, the counter should not change.

In [53]:
try:
    Task("")
except ValueError as exc:
    print("Expected error:", exc)

print(Task.instances_created)

assert Task.instances_created == 2

Expected error: title cannot be empty
2


# Tutorial Problem 18 — Caching class calls

A metaclass does not have to create a new object every time.

It can choose to reuse a previous instance.

In [54]:
class SimpleCacheMeta(type):
    def __call__(cls, key):
        if not hasattr(cls, "_cache"):
            cls._cache = {}

        if key not in cls._cache:
            cls._cache[key] = super().__call__(key)

        return cls._cache[key]

Use the metaclass with a simple resource class.

In [55]:
class Resource(metaclass=SimpleCacheMeta):
    def __init__(self, key):
        self.key = key

a = Resource("alpha")
b = Resource("alpha")
c = Resource("beta")

print("a is b:", a is b)
print("a is c:", a is c)

a is b: True
a is c: False


The repeated `"alpha"` call returns the same object.

This means class callability can be customized to alter **identity**, not just initialization.

### Problem

Generalize the cache so both positional and keyword arguments participate in the key.

Assume all arguments are hashable.

In [56]:
# Solution

class CachedMeta(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)

        cls._instances = {}

        return cls

    def __call__(cls, *args, **kwargs):
        cache_key = (
            args,
            tuple(sorted(kwargs.items())),
        )

        if cache_key not in cls._instances:
            cls._instances[cache_key] = super().__call__(*args, **kwargs)

        return cls._instances[cache_key]

class Service(metaclass=CachedMeta):
    def __init__(self, name, region="eu"):
        self.name = name
        self.region = region

s1 = Service("payments", region="eu")
s2 = Service("payments", region="eu")
s3 = Service("payments", region="us")

print("s1 is s2:", s1 is s2)
print("s1 is s3:", s1 is s3)

assert s1 is s2
assert s1 is not s3

s1 is s2: True
s1 is s3: False


### Design warning

A real cache raises additional questions:

- when should entries be removed?
- what about unhashable arguments?
- is the cache thread-safe?
- should constructor side effects run only once?
- does reused identity make semantic sense?

Advanced construction hooks are powerful, but hidden behavior can make code harder to reason about.

# Tutorial Problem 19 — Classes as plugin factories

A practical use of class callability is a registry.

The registry stores class objects, then calls them later when an instance is needed.

In [57]:
class Exporter:
    def export(self, value):
        raise NotImplementedError

class TextExporter(Exporter):
    def export(self, value):
        return str(value)

class ReprExporter(Exporter):
    def export(self, value):
        return repr(value)

Store the classes themselves.

In [58]:
registry = {
    "text": TextExporter,
    "repr": ReprExporter,
}

for name, exporter_class in registry.items():
    print(name, "callable:", callable(exporter_class))

text callable: True
repr callable: True


Because the stored objects are classes, the registry can create instances simply by calling them.

### Problem

Write `create_exporter(name)`.

It should:

1. find the registered class;
2. call the class;
3. return the new instance;
4. raise a clear error for unknown names.

In [59]:
# Solution

def create_exporter(name):
    try:
        exporter_class = registry[name]
    except KeyError as exc:
        raise ValueError(f"Unknown exporter: {name!r}") from exc

    return exporter_class()

text_exporter = create_exporter("text")
repr_exporter = create_exporter("repr")

print(text_exporter.export({"x": 1}))
print(repr_exporter.export({"x": 1}))

assert isinstance(text_exporter, TextExporter)
assert isinstance(repr_exporter, ReprExporter)

{'x': 1}
{'x': 1}


This pattern is common in serializers, command systems, plugin architectures, parsers, and dependency injection.

# Tutorial Problem 20 — Accept any callable factory

Sometimes an API should not care whether its factory is:

- a class;
- a function;
- a callable object.

It only needs the factory to satisfy the callable protocol.

In [60]:
def build(factory, *args, **kwargs):
    if not callable(factory):
        raise TypeError("factory must be callable")

    return factory(*args, **kwargs)

First, use a class as the factory.

In [61]:
class Product:
    def __init__(self, name):
        self.name = name

product = build(Product, "keyboard")

print(product.name)

keyboard


Now use a normal function.

In [62]:
def uppercase(text):
    return text.upper()

print(build(uppercase, "python"))

PYTHON


Finally, use a callable object.

In [63]:
class Prefixer:
    def __init__(self, prefix):
        self.prefix = prefix

    def __call__(self, text):
        return f"{self.prefix}{text}"

prefix_id = Prefixer("ID-")

print(build(prefix_id, "42"))

ID-42


### Problem

Verify all three factory styles and reject a non-callable.

In [64]:
# Solution

assert isinstance(build(Product, "mouse"), Product)
assert build(uppercase, "hello") == "HELLO"
assert build(prefix_id, "007") == "ID-007"

try:
    build(123, "ignored")
except TypeError as exc:
    print("Expected error:", exc)
else:
    raise AssertionError("Expected TypeError")

Expected error: factory must be callable


The general abstraction is not "function" or "class".

The abstraction is simply **callable**.

# Tutorial Problem 21 — A callable pipeline

Now combine several ideas.

A pipeline will store callables and pass a value through them one after another.

In [65]:
def strip_text(text):
    return text.strip()

def lower_text(text):
    return text.lower()

print(lower_text(strip_text("  PYTHON  ")))

python


Instead of manually nesting calls, we can store the steps.

In [66]:
steps = [
    strip_text,
    lower_text,
]

value = "  ADVANCED PYTHON  "

for step in steps:
    value = step(value)

print(value)

advanced python


Now make the pipeline itself callable.

In [67]:
class Pipeline:
    def __init__(self, *steps):
        for index, step in enumerate(steps):
            if not callable(step):
                raise TypeError(f"step {index} must be callable")

        self.steps = tuple(steps)

    def __call__(self, value):
        for step in self.steps:
            value = step(value)

        return value

### Problem

Add a callable object `ReplaceSpaces("-")` and create this sequence:

```text
strip whitespace
convert to lowercase
replace spaces with hyphens
```

In [68]:
# Solution

class ReplaceSpaces:
    def __init__(self, replacement):
        self.replacement = replacement

    def __call__(self, text):
        return text.replace(" ", self.replacement)

slug_pipeline = Pipeline(
    strip_text,
    lower_text,
    ReplaceSpaces("-"),
)

result = slug_pipeline("  Advanced Python Objects  ")

print(result)

assert result == "advanced-python-objects"
assert callable(slug_pipeline)

advanced-python-objects


The pipeline does not care whether a step is a function, bound method, or callable object.

It only relies on one protocol: the step can be called.

# Tutorial Problem 22 — A class decorator can modify a class object

Metaclasses are not the only way to customize classes.

A class decorator receives the class object after it has been created.

In [69]:
def add_label(cls):
    cls.label = cls.__name__.lower()

    return cls

@add_label
class Report:
    pass

print(Report.label)

report


This works because classes are ordinary Python objects.

A decorator can inspect or modify the class, then return it.

### Problem

Create `add_instance_counter`.

It should wrap `__init__` and count successful initializations.

In [70]:
# Solution

from functools import wraps

def add_instance_counter(cls):
    original_init = cls.__init__

    cls.instances_initialized = 0

    @wraps(original_init)
    def wrapped_init(self, *args, **kwargs):
        original_init(self, *args, **kwargs)

        cls.instances_initialized += 1

    cls.__init__ = wrapped_init

    return cls

@add_instance_counter
class Message:
    def __init__(self, text):
        if not text:
            raise ValueError("text cannot be empty")

        self.text = text

Message("first")
Message("second")

print(Message.instances_initialized)

assert Message.instances_initialized == 2

2


Now verify that a failed `__init__` does not increment the successful initialization count.

In [71]:
try:
    Message("")
except ValueError as exc:
    print("Expected error:", exc)

print(Message.instances_initialized)

assert Message.instances_initialized == 2

Expected error: text cannot be empty
2


A class decorator is often simpler than a metaclass when the customization can happen after the class has already been created.

# Tutorial Problem 23 — Shared construction policy with a metaclass

Suppose several unrelated classes must all produce objects with a positive integer `id`.

A metaclass can enforce that policy after normal construction.

In [72]:
class PositiveIdMeta(type):
    def __call__(cls, *args, **kwargs):
        obj = super().__call__(*args, **kwargs)

        if not isinstance(obj.id, int) or isinstance(obj.id, bool):
            raise TypeError("id must be an integer")

        if obj.id <= 0:
            raise ValueError("id must be positive")

        return obj

Now apply the same rule to two classes.

In [73]:
class Customer(metaclass=PositiveIdMeta):
    def __init__(self, customer_id, name):
        self.id = customer_id
        self.name = name

class Invoice(metaclass=PositiveIdMeta):
    def __init__(self, invoice_id, total):
        self.id = invoice_id
        self.total = total

customer = Customer(1, "Ada")
invoice = Invoice(100, 250.0)

print(customer.__dict__)
print(invoice.__dict__)

{'id': 1, 'name': 'Ada'}
{'id': 100, 'total': 250.0}


### Problem

Verify that zero, negative values, strings, floats, and booleans are rejected appropriately.

In [74]:
# Solution

for bad_id in (0, -1):
    try:
        Customer(bad_id, "Invalid")
    except ValueError as exc:
        print("ValueError:", exc)
    else:
        raise AssertionError("Expected ValueError")

for bad_id in ("1", 1.5, True):
    try:
        Customer(bad_id, "Invalid")
    except TypeError as exc:
        print("TypeError:", exc)
    else:
        raise AssertionError("Expected TypeError")

ValueError: id must be positive
ValueError: id must be positive
TypeError: id must be an integer
TypeError: id must be an integer
TypeError: id must be an integer


Even though `bool` is technically a subclass of `int`, the domain rule deliberately rejects it.

This is a good reminder that programming-language type relationships and application-domain rules are not always identical.

# Tutorial Problem 24 — Validate a class at class-creation time

Metaclasses can customize more than instance creation.

They can also inspect a class while that class object is being created.

In [75]:
class HandlerMeta(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)

        if name != "BaseHandler":
            handler = getattr(cls, "handle", None)

            if not callable(handler):
                raise TypeError(
                    f"{name} must define callable handle()"
                )

        return cls

Create a base class and one valid subclass.

In [76]:
class BaseHandler(metaclass=HandlerMeta):
    pass

class DoubleHandler(BaseHandler):
    def handle(self, value):
        return value * 2

handler = DoubleHandler()

print(handler.handle(5))

assert handler.handle(5) == 10

10


The metaclass checked `DoubleHandler` while the class itself was being created.

No instance validation was needed.

### Problem

Attempt to define a subclass whose `handle` attribute is an integer.

Catch the error around the class definition.

In [77]:
# Solution

try:
    class BrokenHandler(BaseHandler):
        handle = 123
except TypeError as exc:
    print("Class creation failed:", exc)
else:
    raise AssertionError("Expected TypeError during class creation")

Class creation failed: BrokenHandler must define callable handle()


The error occurs during creation of the class object itself.

This is different from metaclass `__call__`, which runs later when the class object is called to construct an instance.

# Tutorial Problem 25 — Capstone: configurable command objects

We will finish with a small command system.

Each command is a callable object.

A dispatcher stores commands by name and invokes them.

In [78]:
class Add:
    def __init__(self, amount):
        self.amount = amount

    def __call__(self, value):
        return value + self.amount

class Multiply:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, value):
        return value * self.factor

Create a command table.

Notice that the table stores instances, not classes.

In [79]:
commands = {
    "plus_10": Add(10),
    "double": Multiply(2),
}

for name, command in commands.items():
    print(name, "callable:", callable(command))

plus_10 callable: True
double callable: True


Now write a dispatcher.

The dispatcher only needs to know that the selected object can be called.

In [80]:
def dispatch(name, value):
    try:
        command = commands[name]
    except KeyError as exc:
        raise ValueError(f"Unknown command: {name!r}") from exc

    return command(value)

Test individual commands first.

In [81]:
print(dispatch("plus_10", 5))
print(dispatch("double", 5))

assert dispatch("plus_10", 5) == 15
assert dispatch("double", 5) == 10

15
10


### Problem

Apply this sequence:

```text
plus_10
double
plus_10
```

to the starting value `5`.

In [82]:
# Solution

sequence = [
    "plus_10",
    "double",
    "plus_10",
]

value = 5

for command_name in sequence:
    value = dispatch(command_name, value)

print("final value:", value)

assert value == 40

final value: 40


The trace is:

```text
5
+ 10 -> 15
* 2  -> 30
+ 10 -> 40
```

The command system demonstrates the broader lesson of this notebook: many different Python objects can cooperate through the callable protocol.

# Final Review

The central idea is simple:

> A Python class is an object, and class objects are normally callable.

Many advanced object-model behaviors become easier to understand once that idea is clear.

## Class calls and instances

For an ordinary class:

```python
obj = MyClass(...)
```

calls the class object and normally produces an initialized instance.

The construction process typically involves:

```text
metaclass __call__
        |
        v
class __new__
        |
        v
class __init__
        |
        v
resulting instance
```

## Class namespace vs instance namespace

Class attributes are shared through the class.

Instance attributes normally represent per-object state.

An instance attribute can shadow a class attribute of the same name.

## `type()` and `isinstance()`

Use:

```python
type(obj)
```

when exact runtime type matters.

Use:

```python
isinstance(obj, BaseClass)
```

for normal inheritance-aware checks.

## Callable instances

Implement:

```python
def __call__(self, ...):
    ...
```

when an object should behave like a configured or stateful function.

## `__new__` and `__init__`

A practical mental model is:

```text
__new__  -> create or return the object
__init__ -> initialize the object
```

Most ordinary classes only need `__init__`.

`__new__` becomes especially useful for immutable subclasses or specialized construction.

## Metaclasses

A metaclass is the type of a class.

For most ordinary classes:

```python
type(MyClass) is type
```

A custom metaclass can influence:

- creation of the class object;
- what happens when the class object is called.

## Best practices

Prefer the simplest abstraction that communicates the design clearly.

Often:

- `__init__` is better than custom `__new__`;
- a named factory is clearer than a constructor returning unrelated types;
- a class decorator is simpler than a metaclass;
- explicit state is easier to reason about than hidden global or class-level mutable state;
- `isinstance()` is better than exact-type checks for polymorphic APIs;
- `callable()` is useful when an API accepts functions, classes, or callable objects interchangeably.

# Additional Advanced Practice — No Solutions Yet

Use the solved tutorial sections above as models.

Try to solve these by breaking each one into small experiments rather than writing the complete solution immediately.

### Practice A — Retry policy

Create:

```python
retry = Retry(max_attempts=3)
```

Then support:

```python
retry(some_function, arg1, arg2)
```

Retry only when the function raises an exception.

### Practice B — Weak-reference instance cache

Replace a normal metaclass cache with `weakref.WeakValueDictionary`.

Observe what happens after every strong reference to a cached instance disappears.

### Practice C — Subclass-safe cache

Use one caching metaclass for both:

```python
ApiClient
AdminClient
```

Identical constructor arguments should reuse instances within one class, but never across the two classes.

### Practice D — Construction timer

Create a metaclass whose `__call__` measures how long successful construction takes.

Store the elapsed duration on the created instance.

### Practice E — Typed registry

Build a registry that accepts only subclasses of a required base class.

Reject:

- instances;
- unrelated classes;
- arbitrary functions.

### Practice F — Validation pipeline

Create several validator objects implementing `__call__`.

A validation pipeline should stop at the first failure and report which validator rejected the value.

### Practice G — Immutable `Vector2D`

Subclass `tuple` and support:

```python
Vector2D(x, y)
vector.x
vector.y
vector.length
```

Construct the immutable tuple contents inside `__new__`.

### Practice H — Predict the exact call order

Before running this code, predict the output:

```python
class Meta(type):
    def __call__(cls):
        print("META")
        return super().__call__()

class Demo(metaclass=Meta):
    def __new__(cls):
        print("NEW")
        return super().__new__(cls)

    def __init__(self):
        print("INIT")

Demo()
```

Then change `__new__` so it returns an integer.

Predict the new output before testing it.

# End of Notebook

For each new object-model problem, use the same workflow:

1. state the expected behavior;
2. build the smallest experiment;
3. inspect the relevant namespace or callable;
4. make a prediction;
5. implement the solution;
6. add assertions;
7. explain why Python produced that result.

That approach is more valuable than memorizing isolated dunder methods.